# Notebook 07: Uncertainty and R-AUC

**Purpose**: Pass-2 metrics. Compute R-AUC and F1@95% using normalised logprob confidence. Runs on results already collected in Notebooks 02 and 06.

## Why R-AUC instead of just reporting accuracy

Accuracy alone hides a specific failure mode this project cares about: a model that's accurate on average but **confidently wrong** on exactly the OOD rows demonstration design is supposed to help with would look identical, on an accuracy-only readout, to a model that's honestly uncertain when it's OOD. The spec adopts the joint robustness-and-uncertainty evaluation paradigm of Malinin et al. (2021, "Shifts," Lit-review §3, ref [32]) specifically to close that gap: **R-AUC** traces an error-retention curve — progressively replacing the model's *least* confident predictions with ground truth, in order of increasing confidence, and integrating the resulting error curve — so it rewards both being right and knowing when you're likely wrong. A model that is accurate but poorly calibrated (confidently wrong on shifted inputs) scores worse on R-AUC than one with the same accuracy but well-calibrated confidence, even though accuracy alone couldn't distinguish them. F1@95% is the complementary, more interpretable number: macro-F1 restricted to just the 95% most confident predictions, which answers "how good are this model's *most trustworthy* predictions specifically."

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

In [2]:
import numpy as np
import pandas as pd

from src.utils.results_schema import load_results
from src.evaluation.uncertainty import confidence_from_logprobs, compute_rauc, compute_f1_at_retention


def try_load(path):
    try:
        return load_results(resolve_path(path))
    except FileNotFoundError:
        return None


real_results = try_load('results/real_arm_baselines.parquet')
synthetic_results = try_load('results/synthetic_evaluation.parquet')

loaded = [df for df in [real_results, synthetic_results] if df is not None]
all_results = pd.concat(loaded, ignore_index=True) if loaded else pd.DataFrame(
    columns=['arm', 'dataset', 'environment', 'model', 'method', 'seed', 'query_id',
             'prediction', 'label', 'logprob_0', 'logprob_1', 'demo_ids', 'k']
)
print(f"Loaded {len(all_results)} prediction rows "
      f"(real: {0 if real_results is None else len(real_results)}, "
      f"synthetic: {0 if synthetic_results is None else len(synthetic_results)})")

Loaded 0 prediction rows (real: 0, synthetic: 0)


## R-AUC and F1@95%

Confidence = `max(exp(logprob_0), exp(logprob_1))` per prediction — already derivable from the logprob_0/logprob_1 columns saved in Notebooks 02/06.

This only works cleanly because Notebook 02/06's constrained single-token decoding (`temperature=0`, forced choice between exactly the two label tokens) guarantees `logprob_0`/`logprob_1` are always directly comparable, constrained-softmax probabilities — not raw next-token logprobs over the model's full vocabulary, which wouldn't be comparable across conditions or even across queries the same way.

In [3]:
UNCERTAINTY_COLS = ['arm', 'dataset', 'environment', 'model', 'method', 'rauc', 'f1_at_95', 'n']
uncertainty_rows = []
retention_curves = {}  # (arm, dataset, environment, model, method) -> (retention_levels, error_curve)

for keys, group in all_results.groupby(['arm', 'dataset', 'environment', 'model', 'method']):
    confidences = confidence_from_logprobs(group['logprob_0'].to_numpy(), group['logprob_1'].to_numpy())
    predictions = group['prediction'].to_numpy()
    labels = group['label'].to_numpy()

    rauc, retention_levels, error_curve = compute_rauc(predictions, labels, confidences)
    f1_95 = compute_f1_at_retention(predictions, labels, confidences, retention=0.95)

    arm, dataset, environment, model, method = keys
    uncertainty_rows.append({
        'arm': arm, 'dataset': dataset, 'environment': environment, 'model': model, 'method': method,
        'rauc': rauc, 'f1_at_95': f1_95, 'n': len(group),
    })
    retention_curves[keys] = (retention_levels, error_curve)

uncertainty_df = pd.DataFrame(uncertainty_rows, columns=UNCERTAINTY_COLS)
uncertainty_df

,arm,dataset,environment,model,method,rauc,f1_at_95,n


In [4]:
import matplotlib.pyplot as plt

resolve_path('results').mkdir(parents=True, exist_ok=True)
uncertainty_df.to_parquet(resolve_path('results/uncertainty_metrics.parquet'), index=False)

# Retention curves for the best (lowest R-AUC) vs worst (highest R-AUC)
# condition, per arm.
resolve_path('figures').mkdir(parents=True, exist_ok=True)
for arm, arm_df in uncertainty_df.groupby('arm'):
    if arm_df.empty:
        continue
    best_key = tuple(arm_df.loc[arm_df['rauc'].idxmin(), ['arm', 'dataset', 'environment', 'model', 'method']])
    worst_key = tuple(arm_df.loc[arm_df['rauc'].idxmax(), ['arm', 'dataset', 'environment', 'model', 'method']])

    fig, ax = plt.subplots()
    for key, label in [(best_key, 'best (lowest R-AUC)'), (worst_key, 'worst (highest R-AUC)')]:
        levels, errors = retention_curves[key]
        ax.plot(levels, errors, label=f"{label}: {key[4]} / {key[1]}")
    ax.set_xlabel('Retention')
    ax.set_ylabel('Error rate')
    ax.set_title(f'Retention curves ({arm} arm)')
    ax.legend()
    fig.savefig(resolve_path('figures') / f'retention_curves_{arm}.pdf')
    plt.close(fig)

print(f"Saved uncertainty_metrics.parquet ({len(uncertainty_df)} rows) + retention curve figures.")

Saved uncertainty_metrics.parquet (0 rows) + retention curve figures.


## Output

- `results/uncertainty_metrics.parquet` — R-AUC and F1@95% per condition
- Retention curve plots for key comparisons